In [1]:
from torch import nn
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
import pytorch_lightning as pl
import segmentation_models_pytorch as smp
from torch import optim
import ezomero

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from pytorch_lightning.callbacks import ModelCheckpoint

import os
from pathlib import Path
import tqdm
from skimage import io
import pandas as pd

W0424 12:44:33.200000 23392 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [2]:
torch.set_float32_matmul_precision('medium')

In [3]:
torch.cuda.is_available()

True

## Define model

In [ ]:

import lightning as pl
import segmentation_models_pytorch as smp
from torch import nn, optim
import torch.nn.functional as F
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2

class FocusLightningModel(pl.LightningModule):
    def __init__(self):
        super(FocusLightningModel, self).__init__()

        # Load the pre-trained ResNet18 model
        unet = smp.Unet(
            "resnet18",
            encoder_weights="imagenet",
            in_channels=1,
            classes=1,
            activation=None,
        )

        # Use the convolutional part as the encoder
        self.encoder = unet.encoder

        # Your custom layers for the regression task
        self.fc1 = nn.Linear(392000, 256)
        self.fc2 = nn.Linear(256, 1)

        self.criterion = nn.MSELoss()

    def forward(self, x):
        x = self.encoder(x)  # use only results from layers 1 and 2 ([1:3])
        flattened_features = [f.view(f.size(0), -1) for f in x[1:3]]
        x = torch.cat(flattened_features, dim=1)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

    def training_step(self, batch, batch_idx):
        images, labels = batch["image"], batch["label"]
        outputs = self(images)
        loss = self.criterion(outputs.squeeze(), labels.float())
        self.log("train_loss", loss)
        return {"loss": loss}

    def validation_step(self, batch, batch_idx):
        images, labels = batch["image"], batch["label"]
        outputs = self(images)
        loss = self.criterion(outputs.squeeze(), labels.float())

        if not hasattr(self, "val_outputs"):
            self.val_outputs = []
        self.val_outputs.append(
            {
                "val_loss": loss,
            }
        )
        return {"val_loss": loss}

    def on_validation_epoch_end(self):
        avg_loss = torch.stack([x["val_loss"] for x in self.val_outputs]).mean()
        self.log("val_loss", avg_loss, prog_bar=True)

        # Optionally clear the outputs to free up memory
        del self.val_outputs

    def test_step(self, batch, batch_idx):
        images, labels = batch["image"], batch["label"]
        outputs = self(images)
        loss = self.criterion(outputs.squeeze(), labels.float())
        self.log("test_loss", loss)

    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=0.001)
        return optimizer


class FocusCNN(nn.Module):
    def __init__(self):
        super(FocusCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(128 * 17 * 17, 256)
        self.fc2 = nn.Linear(256, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv3(x))
        x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


class FocusModel(pl.LightningModule):
    def __init__(self):
        super(FocusModel, self).__init__()
        self.model = FocusCNN()
        self.criterion = nn.MSELoss()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, labels = batch["image"], batch["label"]
        outputs = self(images)
        loss = self.criterion(outputs.squeeze(), labels.float())
        self.log("train_loss", loss)
        return {"loss": loss}

    def validation_step(self, batch, batch_idx):
        images, labels = batch["image"], batch["label"]
        outputs = self(images)
        loss = self.criterion(outputs.squeeze(), labels.float())

        if not hasattr(self, "val_outputs"):
            self.val_outputs = []
        self.val_outputs.append(
            {
                "val_loss": loss,
            }
        )
        return {"val_loss": loss}

    def on_validation_epoch_end(self):
        avg_loss = torch.stack([x["val_loss"] for x in self.val_outputs]).mean()
        self.log("val_loss", avg_loss, prog_bar=True)

        # Optionally clear the outputs to free up memory
        del self.val_outputs

    def test_step(self, batch, batch_idx):
        images, labels = batch["image"], batch["label"]
        outputs = self(images)
        loss = self.criterion(outputs.squeeze(), labels.float())
        self.log("test_loss", loss)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.model.parameters(), lr=0.001)
        return optimizer


def preprocess_stack(stack):
    augmentations = A.Compose(
        [
            A.RandomRotate90(p=0.5),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Transpose(p=0.5),
            A.ShiftScaleRotate(p=0.5),
            A.Normalize(mean=0, std=1, max_pixel_value=255.0),
            A.Resize(height=140, width=140),
            ToTensorV2(),
        ]
    )

    processed_stack = []
    for slice in stack:
        # If grayscale, repeat the single channel three times to simulate RGB
        processed_slice = augmentations(image=slice)["image"]
        processed_stack.append(processed_slice)

    return torch.stack(processed_stack)


def get_focus_plane(model, image_stack):
    import numpy as np

    image_stack = preprocess_stack(image_stack).to("cuda")
    with torch.no_grad():
        offsets = model(image_stack)
        offsets = offsets.squeeze().detach().cpu().numpy()

    # retrieve focus plane
    return np.argmin(abs(offsets))


def detect_focus_plane(
    image: "napari.types.ImageData",
    segmentation: "napari.types.LabelsData",
    width: float,
    height: float,
    scale_xy: float,
    crop_margin: int = 0,
    focus_detector: torch.nn.Module = None,
) -> "napari.types.PointsData":
    from .utilities import crop_well_from_image, get_well_centroids
    import tqdm
    import numpy as np

    well_centroids = get_well_centroids(segmentation) / scale_xy
    focus_planes = []
    for idx in tqdm.tqdm(range(len(well_centroids)), desc="Detecting focus planes"):
        try:
            well = crop_well_from_image(image, well_centroids[idx], width, height)
            well = well[:, crop_margin:-crop_margin, crop_margin:-crop_margin]

            focus_plane = get_focus_plane(focus_detector, well)
            focus_planes.append(focus_plane)
        except:
            focus_planes.append(np.nan)

    focus_planes = np.asarray(focus_planes)
    focus_points = np.concatenate((focus_planes[:, None], well_centroids), axis=1)

    return focus_points


In [5]:
working_directory = r'E:\UserData\Johannes\20260424\focus_detector_training'
images_dir = os.path.join(working_directory, 'images')
labels_dir = os.path.join(working_directory, 'labels')
os.makedirs(images_dir, exist_ok=True)
os.makedirs(labels_dir, exist_ok=True)

## Get training data from OMERO

In [5]:
host = 'omero-nfdi.uni-muenster.de'
port = 4064
user = 'JohannesSoltwedel'
group = None
secure=True
password = "BYMCfyZpMDLC"

conn = ezomero.connect(host=host, user=user, secure=secure, port=port, group=group, password=password)

In [6]:
dataset_id = 301

In [6]:
img_ids = ezomero.get_image_ids(conn, dataset=dataset_id)  # replace 259 with the id of the dataset you want to use
img_ids = [i for i in tqdm.tqdm(img_ids) if len(ezomero.get_map_annotation_ids(conn, object_type="Image", object_id=i)) > 0]  # filter images that have kv-annotation

100%|██████████| 396/396 [00:57<00:00,  6.92it/s]


In [20]:
df_annotation = pd.DataFrame(columns=["File", "z-slice"])
for img_id in tqdm.tqdm(img_ids):
    image_wrapper, image_stack = ezomero.get_image(conn, img_id, dim_order="tczyx")
    filename = os.path.join(images_dir, f"{img_id}.tif")
    io.imsave(filename, image_stack.astype("uint16").squeeze(), check_contrast=False)

    annotation_id = ezomero.get_map_annotation_ids(conn, object_type="Image", object_id=img_id)[0]
    annotation = ezomero.get_map_annotation(conn, annotation_id)

    _df = pd.DataFrame([{"File": Path(filename).stem + ".tif", "z-slice": annotation["focus_plane"]}])
    df_annotation = pd.concat((df_annotation, _df), ignore_index=True)

df_annotation.to_csv(os.path.join(labels_dir, "annotations.csv"), index=False)

100%|██████████| 56/56 [00:52<00:00,  1.06it/s]


## Dataset

In [6]:
class FocusDataset(Dataset):
    def __init__(self, dataframe, augment=True):
        self.dataframe = dataframe
        self.augment = augment

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx, self.dataframe.columns.get_loc("File")]
        annotated_slice = self.dataframe.iloc[idx, self.dataframe.columns.get_loc("z-slice")]

        img = io.imread(img_path)
        total_slices = img.shape[0]

        # Choose a random z-slice from the stack
        random_z_slice = torch.randint(0, total_slices, (1,)).item()

        # Calculate relative position to the annotated z-slice
        relative_position = abs(random_z_slice - annotated_slice)

        if self.augment:
            augmentations = A.Compose([
                A.RandomRotate90(p=0.5),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.Transpose(p=0.5),
                A.Normalize(mean=0, std=1, max_pixel_value=255.0),
                A.Resize(height=140, width=140),
                ToTensorV2()
            ])

            image = augmentations(image=img[random_z_slice])['image']

        else:
            image = img[random_z_slice]

        sample = {
            'image': image,
            'label': relative_position,
            'File': img_path,
        }

        return sample


In [7]:
df = pd.read_csv(os.path.join(labels_dir, "annotations.csv"))

# add root to full file path
df.File = [os.path.join(working_directory, 'images', f) for f in df.File]

# Splitting the dataset into training (70%) and temporary set (30%)
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)

# Splitting the temporary set into validation (2/3 of 30% = 20%) and test (1/3 of 30% = 10%)
val_df, test_df = train_test_split(temp_df, test_size=0.33, random_state=42)

# Now, you can use these DataFrames to create separate dataset instances:
train_dataset = FocusDataset(train_df)
val_dataset = FocusDataset(val_df)
test_dataset = FocusDataset(test_df)

## Training

This will generate a folder for each training run next to this notebook ('logs_focus_detector'), where you'll find the checkpoint model file for further usage. For viosualixzation, install tensorboard like this:
'''bash
pip install tensorboard
'''

Then, open a new miniforge prompt, activate the environment and type

'''bash
tensorboard --logdir=path/to/log/directory
'''

and navigate to [this page](http://localhost:6006/) in your browser



In [8]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

In [9]:
model = FocusLightningModel()

### Checkpoints and callbacks

In [10]:

# Setting up Model Checkpointing
checkpoint_callback = ModelCheckpoint(
    save_top_k=1,  # Save the best model only and call it best.pt
    save_last=True,  # Also save the last model
    verbose=True,
    monitor='val_loss',  # Assuming you have a validation loss logged with this name
    mode='min',  # Save the model with the minimum 'val_loss',
    filename='best'
)

In [11]:
trainer = pl.Trainer(
    max_epochs=200,
    default_root_dir='./logs_focus_detector',  # Directory for TensorBoard logs
    callbacks=[checkpoint_callback])

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [12]:
trainer.fit(model, train_loader, val_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder   │ ResNetEncoder │ 11.2 M │ train │     0 │
│ 1 │ fc1       │ Linear        │  100 M │ train │     0 │
│ 2 │ fc2       │ Linear        │    257 │ train │     0 │
│ 3 │ criterion │ MSELoss       │      0 │ train │     0 │
└───┴───────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 111 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 111 M                                                                                                
Total estimated model params size (MB): 446                                                                        
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\Leica-User\miniforge3\envs\dl\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

...

Epoch 196, global step 197: 'val_loss' was not in top 1


Epoch 197, global step 198: 'val_loss' was not in top 1


Epoch 198, global step 199: 'val_loss' was not in top 1


Epoch 199, global step 200: 'val_loss' reached 1.27311 (best 1.27311), saving model to 'logs_focus_detector\\lightning_logs\\version_6\\checkpoints\\best.ckpt' as top 1


`Trainer.fit` stopped: `max_epochs=200` reached.
